In [1]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

MODEL_PATH = "pose_landmarker.task"

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    min_pose_detection_confidence=0.1,
    min_pose_presence_confidence=0.1,
    min_tracking_confidence=0.1
)
detector = vision.PoseLandmarker.create_from_options(options)



I0000 00:00:1778171523.625331   19278 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778171523.677968   19280 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778171523.692940   19280 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [2]:
models = [

    "gemma4:e4b",
    "llava:7b"
    ]
    

In [3]:
ids=[0,2,3,4,5,6,7,8]
actuals=[8.5,7,7,6,8.5,6,7]

In [4]:
from utils import *

In [5]:
#pip install ollama
import ollama

In [6]:
from pydantic import BaseModel, Field
class ThrowCritique(BaseModel):
    score: int = Field(ge=1, le=10)
    feedback: str = Field(
        description="Brief coaching feedback in 2-4 sentences."
    )

In [7]:
throwprompt2 = """
You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = strong collegiate athlete
10 = world class technique

Give 2-4 sentences of specific, actionable feedback.
"""

throwprompt3 = """
You are a discus coach whose job is to identify the single biggest flaw in a throw.

Look across the full sequence of images and determine:
1. The overall score from 1-10
2. The biggest mistake hurting the throw
3. The one change that would improve the throw the most

Keep the feedback concise and concrete. Do not mention uncertainty unless absolutely necessary.
"""



throwprompt5 = """
You are a brutally honest but helpful throwing coach.

Analyze the series of images as if you are watching a video of the throw. Pay close attention to:
- whether the athlete stays balanced
- whether the hips lead the shoulders
- whether the throwing arm lags correctly
- whether the release appears high and powerful

Give a score from 1-10. Then provide feedback in the style of a coach talking directly to the athlete: short, blunt, and practical.
"""

In [8]:
def critique_throw(image_paths: list,
                   mod: str = "llava:7b",
                   prompt: str = throwprompt2,  
                   temp: float = 0.0):
    
    resp = ollama.chat(
        model=mod,
        messages=[
            {
                "role": "system",
                "content": prompt,
                "images": image_paths
            }
        ],
        format=ThrowCritique.model_json_schema(),
        options={"temperature": temp}
    )

    raw = resp["message"]["content"]
    return ThrowCritique.model_validate_json(raw)

In [9]:
prompts=[throwprompt2, throwprompt3, throwprompt5]

In [10]:
import time
import pandas as pd

In [11]:
frames = sample_frames_from_mp4("videoplayback.mp4",10)
overs=pose_overlays_from_frames(frames,detector)
img_paths=write_temp_images_for_llm(overs)

crit=critique_throw(image_paths=img_paths,mod='gemma4:e4b',prompt=prompts[2])
crit

W0000 00:00:1778171524.222197   19288 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


ThrowCritique(score=6, feedback="You're losing your balance on the follow-through. You need to drive through your front leg and keep your core tight. Don't let your upper body collapse. Focus on a powerful, stable finish.")

In [12]:
def doscore(count, instructions, mod):
    
    scores=[]
    durations=[]
    feedbacks=[]
    names=[]

    for i in range(1, 8):
        
        idstr = "" if i == 1 else f"-{i}"
        filename = f"videoplayback{idstr}.mp4"
        names.append(filename)
    
    #for i in range(1,13):
    #    idstr = f"{i}"
    #    filename = f"throw{idstr}.mp4"
    #    names.append(filename)

    for file in names:
        
        start = time.time()

        try:
            frames = sample_frames_from_mp4(file, count)
            overs = pose_overlays_from_frames(frames, detector)
            img_paths = write_temp_images_for_llm(overs)
            crit = critique_throw(image_paths=img_paths, mod=mod, prompt=instructions)
            scores.append(crit.score)
            feedbacks.append(crit.feedback)
            
        except Exception as e:
            print(f"Failed on {file}: {e}")
            scores.append(pd.NA)
            feedbacks.append(pd.NA)
            names.append(file)

        durations.append(time.time() - start)

    return (
        pd.DataFrame({"score": scores, "duration": durations,
                      "feedback":feedbacks,"filenames":names}).\
            assign(frames=count, prompt=instructions[:20], model=mod)
    )

In [13]:
report=pd.DataFrame()
for j in prompts:
    for k in models:
        df=doscore(18,j,k)
        report=pd.concat([report,df])
        print(j,k)


You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = strong collegiate athlete
10 = world class technique

Give 2-4 sentences of specific, actionable feedback.
 gemma4:e4b

You are an elite Olympic discus coach and biomechanist.

You have been given a sequence of images showing a complete discus throw from beginning to end.

Evaluate the thrower's technique using these criteria:
- balance and posture
- rotation mechanics
- footwork
- hip and shoulder separation
- release position
- follow-through

Be strict but fair. Rate the throw from 1-10 where:
1 = very poor form
5 = average high school athlete
8 = stron

In [14]:
report.to_csv("wannoreport_may.csv")

In [15]:
report

,score,duration,feedback,filenames,frames,prompt,model
0,7,14.708393,The thrower demonstrates solid power generatio...,videoplayback.mp4,18,\nYou are an elite Ol,gemma4:e4b
1,7,14.272407,The thrower demonstrates solid power generatio...,videoplayback-2.mp4,18,\nYou are an elite Ol,gemma4:e4b
2,7,14.244928,The thrower demonstrates solid power generatio...,videoplayback-3.mp4,18,\nYou are an elite Ol,gemma4:e4b
3,7,14.243802,The thrower demonstrates solid power generatio...,videoplayback-4.mp4,18,\nYou are an elite Ol,gemma4:e4b
4,7,14.232394,The thrower demonstrates solid power generatio...,videoplayback-5.mp4,18,\nYou are an elite Ol,gemma4:e4b
5,7,14.264203,The thrower demonstrates solid power generatio...,videoplayback-6.mp4,18,\nYou are an elite Ol,gemma4:e4b
6,7,14.251058,The thrower demonstrates solid power generatio...,videoplayback-7.mp4,18,\nYou are an elite Ol,gemma4:e4b
0,7,19.763068,The thrower in the image demonstrates a solid ...,videoplayback.mp4,18,\nYou are an elite Ol,llava:7b
1,7,18.536069,The thrower in the image demonstrates a good l...,videoplayback-2.mp4,18,\nYou are an elite Ol,llava:7b
2,7,18.143343,The thrower in the image demonstrates a good b...,videoplayback-3.mp4,18,\nYou are an elite Ol,llava:7b


In [16]:
import os
import time

while True:
    os.system('say "I am done now"')
    time.sleep(2)  # wait 2 seconds before saying it again

KeyboardInterrupt: 